# Experimento Federado Semântico IoT/GenAI 6G

Neste laboratório em Jupyter Notebook, simulamos uma arquitetura de comunicação Edge via redes semânticas.

## 📖 Sumário Executivo: A Bússola do Experimento

Para fins de validação acadêmica e facilidade de reprodução, este Jupyter Notebook atua como o **Painel de Controle Central** de toda a simulação semântica 6G. Cada módulo possui um propósito estrito e eles **devem ser executados na ordem numérica apresentada**:

> [!NOTE]  
> **Regra de Ouro:** Não avance para um módulo subsequente enquanto a barra de progresso (Loader) da célula atual não indicar `[100% concluído]` ou o aviso de `MERGULHO CONCLUÍDO`.

<br>

| Módulo Estrutural | Descrição Funcional Acadêmica | Obrigatório | Células |
| :--- | :--- | :---: | :---: |
| 📘 **Módulo 1: A Fundação Edge** | Define os parâmetros globais (`DATASET`, `MODEL`), valida a comunicação com os contêineres Docker e **baixa os datasets** (MNIST, Fashion‑MNIST, CIFAR‑10). A **Célula 3 é obrigatória** — sem os dados no volume o treinamento da Célula 4 falha. | ✅ Sim | `1, 2, 3` |
| 🧠 **Módulo 2: O Cérebro Semântico** | Núcleo de aprendizagem federada. A **Célula 4** treina o Autoencoder (canal semântico) via FedAvg e **salva os pesos**. A **Célula 5** treina o Classificador Juiz com esses pesos. Ambos os modelos são reutilizados pelas Células 6 e 7 — **não são retreinados**. | ✅ Sim | `4 e 5` |
| 🔬 **Módulo 3: O Laboratório Visual** | *(Recomendado antes do Paper)* Auditoria ocular. Renderiza mosaicos de amostras reais: original × recebido com ruído × reconstruído pelo Decoder. Usa os pesos fixos da Célula 4. Ideal para validar qualitativamente o canal e capturar figuras para o artigo. | ⚡ Opcional | `6` |
| 🏭 **Módulo 4: A Fábrica de Trade-off** | O Benchmark do *Paper*. **Não retreina nenhum modelo.** Usa os pesos salvos pelas Células 4 e 5 e varia apenas a dimensão latente `[16, 32, 64, 128, 256 bytes]` no canal para mapear a Curva de Trade-off (Compressão × Queda Semântica). Exporta **CSV + 2 gráficos** prontos para o LaTeX. | ✅ Sim | `7` |

## 📘 Módulo 1: A Fundação Edge (Setup do Testbed)
Aqui instanciamos o cérebro PyTorch e subimos os contêineres Docker para emular nós GenAI independentes da nuvem.

In [12]:
# Célula 1 — Setup
import json, subprocess, time
from pathlib import Path
import requests

try:
    import pandas as pd; HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False; print('[aviso] pandas ausente')

try:
    import matplotlib.pyplot as plt, matplotlib, numpy as np
    HAS_MATPLOTLIB = True
    print(f'[ok] matplotlib {matplotlib.__version__}')
except Exception:
    HAS_MATPLOTLIB = False; print('[aviso] matplotlib ausente')

ML_SERVICE  = 'http://localhost:8000'
FL_SERVER   = 'http://localhost:8100'
SHARED_ROOT = Path('shared_data')
RESULTS_DIR = SHARED_ROOT / 'resultados' / 'notebook_monitor'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════
#  CONFIGURAÇÃO CENTRAL
# ════════════════════════════════════════════════════════
DATASET  = 'cifar10'    # mnist | fashion | cifar10
MODEL    = 'cnn_vae'
CLIENTS  = 3
ROUNDS   = 3
EPOCHS   = 3
SEED     = 42
AWGN_CFG = {'enabled': True, 'snr_db': 15}
MASK_CFG = {'enabled': True, 'drop_rate': 0.1, 'fill_value': 0.0}
LATENT_DIM = 256   # Utilizado nas etapas Manuais (Células 4 e 5)

# ════════════════════════════════════════════════════════

_BYTES_BRUTOS  = {'mnist': 3136, 'fashion': 3136, 'cifar10': 12288}
_BYTES_LATENTE = 256*4 
print(f'[ok] {DATASET} | {MODEL} | {CLIENTS} clientes | {ROUNDS} rounds')
print(f'     Reducao teor.: {(1 - _BYTES_LATENTE/_BYTES_BRUTOS[DATASET])*100:.1f}%')

[ok] matplotlib 3.10.8
[ok] cifar10 | cnn_vae | 3 clientes | 3 rounds
     Reducao teor.: 91.7%


In [13]:
# Célula 2 — Subir containers Docker
import json, shutil, subprocess, time
from pathlib import Path
import requests

SHARED_ROOT = Path('shared_data')
for pasta in [SHARED_ROOT / 'fl-weights', SHARED_ROOT / 'resultados']:
    if pasta.exists(): shutil.rmtree(pasta); print(f'[setup] limpo: {pasta}')
for d in [SHARED_ROOT/'ml-data'/'datasets', SHARED_ROOT/'ml-data'/'runs',
          SHARED_ROOT/'ml-data'/'logs', SHARED_ROOT/'fl-weights',
          SHARED_ROOT/'resultados'/'notebook_monitor']:
    d.mkdir(parents=True, exist_ok=True)

print('[setup] Subindo containers...')
res = subprocess.run(['docker', 'compose', 'up', '-d', '--build'], capture_output=True, text=True)
if res.returncode != 0: print('ERRO:', res.stderr[-2000:]); raise RuntimeError('docker compose falhou')
print(f'[setup] OK')
time.sleep(8)
for nome, porta in [('ml-service', 8000), ('fl-server', 8100)]:
    ok = False
    for _ in range(20):
        try:
            if requests.get(f'http://localhost:{porta}/health', timeout=3).ok: ok = True; break
        except: pass
        time.sleep(2)
    print(f'[setup] {nome}: {"OK" if ok else "NAO RESPONDEU"}')
    if not ok: raise RuntimeError(f'{nome} nao respondeu')
print('[setup] Todos os containers prontos!')

[setup] limpo: shared_data\fl-weights
[setup] limpo: shared_data\resultados
[setup] Subindo containers...
[setup] OK
[setup] ml-service: OK
[setup] fl-server: OK
[setup] Todos os containers prontos!


In [14]:
# Célula 3 — Baixar datasets dentro do container
import time, requests
ML_SERVICE = 'http://localhost:8000'
DATASETS_A_BAIXAR = ['mnist', 'fashion', 'cifar10']
print(f'[datasets] Solicitando: {DATASETS_A_BAIXAR}')
r = requests.post(f'{ML_SERVICE}/data/prepare', json={'datasets': DATASETS_A_BAIXAR}, timeout=10)
print(f'[datasets] {r.json()}')
t0, ultimo = time.time(), None
while True:
    try: st = requests.get(f'{ML_SERVICE}/data/prepare/status', timeout=5).json()
    except: time.sleep(3); continue
    if st.get('current') != ultimo and st.get('current'):
        print(f'[datasets] Baixando {st["current"].upper()}...'); ultimo = st['current']
    if not st.get('running'):
        print(f'[datasets] Prontos: {st.get("done")}'); break
    if time.time()-t0 > 600: print('TIMEOUT'); break
    time.sleep(3)
print('[datasets] Pronto!')

[datasets] Solicitando: ['mnist', 'fashion', 'cifar10']
[datasets] {'status': 'started', 'datasets': ['mnist', 'fashion', 'cifar10']}
[datasets] Baixando MNIST...
[datasets] Baixando CIFAR10...
[datasets] Prontos: ['mnist', 'fashion', 'cifar10']
[datasets] Pronto!


---
## 🧠 Módulo 2: Os Dois Cérebros Semânticos (Treinamento Paralelo)

*A Comunicação Semântica exige dois agentes: A Antena Compressora (O Encoder Rádio Base) e o Sensor Decodificador (O Juiz Classificador).* Nossa inteligência artificial deve agora construir as lógicas matemáticas de transmissão para ruídos complexos.


In [ ]:
"""
fix_cell4.py — Célula 4 (Arquitetura Final Unificada)
======================================================
Para cada latent_dim em [16, 32, 64, 128, 256]:
  1. Dispara treinamento do Autoencoder via FedAvg (fl-server/fl-clients)
  2. Monitora progresso em tempo real (logs + status do fl-server)
  3. Aguarda conclusão antes de passar para o próximo dim

Os pesos são salvos automaticamente pelo fl-server como:
  /ml-data/weights/{DATASET}_{MODEL}_d{DIM}.pth

Execute este script no Jupyter com:
  %run fix_cell4.py
ou:
  exec(open('fix_cell4.py').read())
"""

import json
import time
from pathlib import Path

import requests

# ─── Configuração ────────────────────────────────────────────────────────────
ML_SERVICE  = "http://localhost:8000"
FL_SERVER   = "http://localhost:8100"
LOGS_DIR    = Path("shared_data/ml-data/logs")

DATASET     = "cifar10"   # mnist | fashion | cifar10
MODEL       = "cnn_vae"
CLIENTS     = 3
ROUNDS      = 3
EPOCHS      = 3
SEED        = 42
AWGN_CFG    = {"enabled": True, "snr_db": 15}
MASK_CFG    = {"enabled": True, "drop_rate": 0.1, "fill_value": 0.0}

# A dimensão central a ser usada nas células individuais (Células 5 e 6)
# Aqui treinamos TODAS as dimensões.
LATENT_DIMS = [256]

TIMEOUT     = 7200  # 2h min por dim
MAX_FALHAS  = 10
KEYWORDS    = (
    "round", "ep ", "epoch", "loss", "fedavg", "submit",
    "shard", "done", "error", "aggregat", "client", "training",
    "batch", "lr=",
)

# ─── Helpers ─────────────────────────────────────────────────────────────────

def _novos(path: Path, offset: int) -> tuple[str, int]:
    if not path.exists():
        return "", offset
    try:
        t = path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return "", offset
    return t[offset:], len(t)


def _barra(atual: int, total: int, w: int = 30) -> str:
    p = min(atual / max(total, 1), 1.0)
    c = int(p * w)
    return "█" * c + "░" * (w - c)


def _treinar_dim(latent_dim: int) -> dict:
    """Dispara treinamento para um latent_dim e monitora até conclusão."""
    print(f"\n{'='*65}")
    print(f"  ▶ TREINANDO  latent_dim={latent_dim}  [{DATASET}/{MODEL}]")
    print(f"{'='*65}")

    payload = {
        "dataset": DATASET,
        "model": MODEL,
        "clients": CLIENTS,
        "epochs": EPOCHS,
        "latent_dim": latent_dim,
        "rounds": ROUNDS,
        "seed": SEED,
        "compression_mode": "baseline",
        "compression_bits": 8,
        "awgn": AWGN_CFG,
        "masking": MASK_CFG,
    }

    st = requests.post(f"{ML_SERVICE}/training/start", json=payload, timeout=15).json()
    print(f"  [treino] → {st.get('status')}")
    time.sleep(5)

    offs = {"server": 0}
    for i in range(1, CLIENTS + 1):
        offs[f"client-{i}"] = 0

    t0 = time.time()
    falhas = 0
    ur, ue = -1, ""

    while True:
        elapsed = int(time.time() - t0)
        if elapsed > TIMEOUT:
            print("  [TIMEOUT] tempo estourado")
            break

        try:
            ml = requests.get(f"{ML_SERVICE}/training/status", timeout=5).json()
            falhas = 0
        except Exception:
            falhas += 1
            if falhas >= MAX_FALHAS:
                print("  [ERRO] containers não respondem")
                break
            time.sleep(3)
            continue

        try:
            s     = requests.get(f"{FL_SERVER}/training/status", timeout=5).json()
            estado = s.get("state", "?")
            rnd    = int(s.get("current_round") or 0)
            tot    = int(s.get("total_rounds") or ROUNDS)
            sub    = s.get("submitted_clients", [])
            exp    = int(s.get("expected_clients") or CLIENTS)
            hist   = s.get("history", [])
        except Exception:
            estado, rnd, tot, sub, exp, hist = "?", 0, ROUNDS, [], CLIENTS, []

        if rnd != ur or estado != ue:
            barra = _barra(rnd, tot)
            pct   = rnd / max(tot, 1) * 100
            sub_s = ", ".join(str(x) for x in sorted(sub)) if sub else "nenhum"
            print(f"  +-- ROUND [{barra}] {rnd}/{tot} ({pct:.0f}%) [{elapsed}s] {estado}")
            print(f"  |   Submetidos: [{sub_s}] / {exp}")
            if hist:
                h = hist[-1]
                print(f"  |   Round {h.get('epoch', '?'):>2}: loss={h.get('loss', 0):.5f}")
            print(f"  +{'-'*62}")
            ur, ue = rnd, estado

        for i in range(1, CLIENTS + 1):
            k = f"client-{i}"
            nv, offs[k] = _novos(LOGS_DIR / f"training_{k}.log", offs[k])
            for ln in nv.strip().split("\n"):
                if ln.strip() and any(w in ln.lower() for w in KEYWORDS):
                    print(f"  [client-{i}] {ln.strip()}")

        nv_s, offs["server"] = _novos(LOGS_DIR / "training_server.log", offs["server"])
        for ln in nv_s.strip().split("\n"):
            if ln.strip() and any(w in ln.lower() for w in KEYWORDS):
                print(f"  [server  ] {ln.strip()}")

        if not ml.get("running", False):
            try:
                res     = requests.get(f"{ML_SERVICE}/results/latest", timeout=10).json()
                fl      = res.get("final_loss", "?")
                print(f"  {'='*62}")
                print(f"  ✓ latent_dim={latent_dim} CONCLUÍDO em {elapsed//60}min {elapsed%60}s | loss={fl}")
                print(f"  {'='*62}")
                return {"latent_dim": latent_dim, "final_loss": fl, "elapsed": elapsed}
            except Exception as exc:
                print(f"  [erro ao ler resultado] {exc}")
            break

        time.sleep(4)

    return {"latent_dim": latent_dim, "final_loss": "?", "elapsed": int(time.time() - t0)}


# ─── Loop principal ───────────────────────────────────────────────────────────
print("╔" + "═"*67 + "╗")
print("║  CÉLULA 4 — AUTOENCODER × LATENT_DIM (Arquitetura Unificada)     ║")
print("╚" + "═"*67 + "╝")
print(f"  Dataset: {DATASET}  |  Modelo: {MODEL}  |  Dims: {LATENT_DIMS}")
print()

resultados_c4 = []
for dim in LATENT_DIMS:
    r = _treinar_dim(dim)
    resultados_c4.append(r)
    # Pequena pausa entre treinos para o servidor se estabilizar
    time.sleep(3)

print("\n" + "═"*68)
print("  RESUMO — CÉLULA 4")
print("═"*68)
print(f"  {'latent_dim':>12} | {'final_loss':>12} | {'tempo':>8}")
print(f"  {'-'*12}-+-{'-'*12}-+-{'-'*8}")
for r in resultados_c4:
    t = r['elapsed']
    print(f"  {r['latent_dim']:>12} | {str(r['final_loss']):>12} | {t//60}m{t%60:02d}s")

print("\n✓ Célula 4 concluída! Execute fix_cell5.py (Classificadores).")

# Persiste resultado para as próximas células
Path("shared_data/resultados/notebook_monitor").mkdir(parents=True, exist_ok=True)
Path("shared_data/resultados/notebook_monitor/cell4_resultado.json").write_text(
    json.dumps({
        "dataset": DATASET, "model": MODEL,
        "latent_dims": LATENT_DIMS, "resultados": resultados_c4,
    }, indent=2),
    encoding="utf-8",
)


╔═══════════════════════════════════════════════════════════════════╗
║  CÉLULA 4 — AUTOENCODER × LATENT_DIM (Arquitetura Unificada)     ║
╚═══════════════════════════════════════════════════════════════════╝
  Dataset: cifar10  |  Modelo: cnn_vae  |  Dims: [256]


  ▶ TREINANDO  latent_dim=256  [cifar10/cnn_vae]
  [treino] → started
  +-- ROUND [██████████░░░░░░░░░░░░░░░░░░░░] 1/3 (33%) [0s] round_active
  |   Submetidos: [nenhum] / 3
  +--------------------------------------------------------------
  [client-1] [23:45:49] [client-1] Client 1/3 waiting 0s before start
  [client-1] [23:45:49] [client-1] Waiting for fl-server at http://fl-server:8100...
  [client-1] [23:45:49] [client-1] Connected to fl-server
  [client-1] [23:45:49] [client-1] Device=cpu; polling server
  [client-1] [23:47:07] [client-1] Loaded shard [0,16666) size=16666
  [client-1] [23:47:08] [client-1] [round 1 ep 1/3] batch 0/131 loss=0.06296
  [client-1] [23:47:55] [client-1] [round 1 ep 1/3] batch 60/131 loss=0.0

In [ ]:
"""
fix_cell5.py — Célula 5 (Arquitetura Final Unificada)
======================================================
Para cada latent_dim em [16, 32, 64, 128, 256]:
  1. Carrega encoder_{dim}.pth (treinado na Célula 4)
  2. Treina Classificador Juiz sobre as reconstruções DESSE encoder
  3. Salva classifier_{dim}.pth

Raciocínio: o Classificador deve ser calibrado para o nível de qualidade
da reconstrução do encoder que ele vai julgar.  Um Juiz treinado em
latent=32 estaria enviesado se aplicado às reconstruções borradas de latent=16.

Execute com:
  %run fix_cell5.py
"""

import json
import time
from pathlib import Path

import requests

# ─── Configuração ────────────────────────────────────────────────────────────
ML_SERVICE  = "http://localhost:8000"
DATASET     = "cifar10"   # mnist | fashion | cifar10
MODEL       = "cnn_vae"
LATENT_DIMS = [256]

# Configurações de treino do classificador por dataset
CONFIG_TREINO = {
    "mnist":   {"epochs": 4,  "samples": 5000},
    "fashion": {"epochs": 8,  "samples": 12000},
    "cifar10": {"epochs": 20, "samples": 25000},
}

# Endpoint legado que a Célula 5 usa
TRAIN_QUICK_URL = f"{ML_SERVICE}/classifier/train-quick"
STATUS_URL      = f"{ML_SERVICE}/classifier/train-quick/status"

# ─── Helpers ─────────────────────────────────────────────────────────────────

def _treinar_classificador(latent_dim: int, cfg: dict) -> dict:
    """Treina um classificador calibrado para latent_dim e aguarda conclusão."""
    print(f"\n  ▶ [{DATASET.upper()} / latent_dim={latent_dim}]  Acordando API...")
    print(f"    [PARÂMETROS] → {cfg['epochs']} épocas | {cfg['samples']} imagens | dim={latent_dim}")

    payload = {
        "dataset":    DATASET,
        "epochs":     cfg["epochs"],
        "samples":    cfg["samples"],
        "seed":       42,
        "latent_dim": latent_dim,   # chave fundamental: informa ao backend qual encoder usar
    }

    try:
        start = requests.post(TRAIN_QUICK_URL, json=payload, timeout=10)
    except Exception as exc:
        print(f"    [FALHA HTTP] {exc}")
        return {"latent_dim": latent_dim, "accuracy": None, "error": str(exc)}

    if start.status_code not in (200, 202):
        print(f"    [ERRO API] status={start.status_code}: {start.text[:200]}")
        return {"latent_dim": latent_dim, "accuracy": None, "error": start.text[:200]}

    print(f"    [STATUS] Pedido aceito — monitorando...")
    t0 = time.time()
    ultimo_ep = -1

    while True:
        try:
            st = requests.get(STATUS_URL, timeout=5).json()
        except Exception:
            time.sleep(3)
            continue

        ep      = st.get("epoch", 0)
        tot     = st.get("total_epochs", cfg["epochs"])
        elapsed = int(time.time() - t0)

        if ep != ultimo_ep and ep > 0:
            print(f"    ✓ Epoch {ep}/{tot} ({ep/max(tot,1)*100:.0f}%) [{elapsed}s]")
            ultimo_ep = ep

        if not st.get("running"):
            if st.get("error"):
                print(f"    [ERRO TREINO] {st['error']}")
                return {"latent_dim": latent_dim, "accuracy": None, "error": st["error"]}
            elif st.get("done"):
                acc = st.get("accuracy", 0.0)
                print(f"    ★ CONCLUÍDO em {elapsed}s | Acurácia: {acc*100:.1f}%")
                return {"latent_dim": latent_dim, "accuracy": acc, "elapsed": elapsed}
            else:
                print("    [AVISO] Status parou inesperadamente — verifique logs do docker.")
                return {"latent_dim": latent_dim, "accuracy": None, "error": "stopped unexpectedly"}

        if elapsed > 7200:
            print("    [TIMEOUT] 2h estouradas.")
            return {"latent_dim": latent_dim, "accuracy": None, "error": "timeout"}

        time.sleep(4)


# ─── Loop principal ───────────────────────────────────────────────────────────
print("╔" + "═"*67 + "╗")
print("║  CÉLULA 5 — CLASSIFICADOR JUIZ × LATENT_DIM (Calibrado)          ║")
print("╚" + "═"*67 + "╝")
print(f"  Dataset: {DATASET}  |  Dims: {LATENT_DIMS}")
print()

DS_safe = DATASET.lower().strip()
cfg     = CONFIG_TREINO.get(DS_safe, {"epochs": 5, "samples": 3000})

print("=" * 68)
print(f" Escola de Juízes Semânticos — {DS_safe.upper()}")
print("=" * 68)
print(f" Fila: {LATENT_DIMS}\n")

resultados_c5 = []
for dim in LATENT_DIMS:
    r = _treinar_classificador(dim, cfg)
    resultados_c5.append(r)
    time.sleep(2)

print("\n" + "═"*68)
print("  RESUMO — CÉLULA 5")
print("═"*68)
print(f"  {'latent_dim':>12} | {'acurácia':>10} | {'tempo':>8}")
print(f"  {'-'*12}-+-{'-'*10}-+-{'-'*8}")
for r in resultados_c5:
    acc  = f"{r['accuracy']*100:.1f}%" if r.get("accuracy") is not None else "ERRO"
    t    = r.get("elapsed", 0)
    print(f"  {r['latent_dim']:>12} | {acc:>10} | {t//60}m{t%60:02d}s")

print("\n✓ Rede Julgadora Profunda finalizada!")
print("  Execute fix_cell6.py (Mosaico Visual) ou fix_cell7.py (Benchmark).")

# Persiste resultado
Path("shared_data/resultados/notebook_monitor").mkdir(parents=True, exist_ok=True)
Path("shared_data/resultados/notebook_monitor/cell5_resultado.json").write_text(
    json.dumps({
        "dataset": DATASET, "model": MODEL,
        "latent_dims": LATENT_DIMS, "resultados": resultados_c5,
    }, indent=2),
    encoding="utf-8",
)


---
## 🔬 Módulo 3: O Laboratório Visual (Auditoria Humana)

Antes de plotar gráficos em larga escala, precisamos atestar visualmente que as perdas do **Canal de Shannon** realmente machucam a classificação semântica no caso severo.


In [ ]:
"""
fix_cell6.py — Célula 6 (Arquitetura Final Unificada)
======================================================
Para cada latent_dim em [16, 32, 64, 128, 256]:
  1. Carrega encoder_{dim}.pth (treinado na Célula 4)
  2. Carrega classifier_{dim}.pth (calibrado na Célula 5)
  3. Renderiza mosaico: original × recebida × reconstruída + veredito do Juiz

Esta é a FIGURA DO ARTIGO:
  latent=16:  [avião borrado]  → Juiz errou: "gato"  (74% queda semântica)
  latent=32:  [avião OK]       → Juiz acertou         (7% queda semântica)
  latent=64:  [avião nítido]   → Juiz acertou         (X% queda semântica)

Execute com:
  %run fix_cell6.py
"""

import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import requests

matplotlib.rcParams["font.family"]    = "DejaVu Sans"
matplotlib.rcParams["axes.titlesize"] = 8

# ─── Configuração ────────────────────────────────────────────────────────────
ML_SERVICE  = "http://localhost:8000"
DATASET     = "cifar10"        # mnist | fashion | cifar10
MODEL       = "cnn_vae"
LATENT_DIMS = [256]
N_AMOSTRAS  = 3                # imagens por dimensão

AWGN_CFG = {"enabled": True,  "snr_db": 15}
MASK_CFG = {"enabled": True,  "drop_rate": 0.1, "fill_value": 0.0}

OUT_DIR = Path("shared_data/resultados/mosaico_visual")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Labels humanos por dataset
_LABELS = {
    "mnist":   {i: str(i) for i in range(10)},
    "fashion": {
        0: "camiseta", 1: "calça",    2: "suéter",  3: "vestido",
        4: "casaco",   5: "sandália", 6: "camisa",   7: "tênis",
        8: "bolsa",    9: "bota",
    },
    "cifar10": {
        0: "avião",    1: "carro",    2: "pássaro", 3: "gato",
        4: "veado",    5: "cachorro", 6: "sapo",    7: "cavalo",
        8: "navio",    9: "caminhão",
    },
}
LABEL_MAP = _LABELS.get(DATASET, {i: str(i) for i in range(10)})

# ─── Helpers ─────────────────────────────────────────────────────────────────

def _buscar_amostras(latent_dim: int, n: int) -> list[dict]:
    """Busca N amostras via /semantic/process usando o encoder do latent_dim."""
    resultados = []
    for _ in range(n * 4):
        try:
            r = requests.post(
                f"{ML_SERVICE}/semantic/process",
                json={
                    "dataset":      DATASET,
                    "model_type":   MODEL,
                    "latent_dim":   latent_dim,
                    "bits":         8,
                    "base_weights": "latest",
                    "awgn":         AWGN_CFG,
                    "masking":      MASK_CFG,
                    "classifier":   {"enabled": True, "top_k": 1, "min_confidence": 0.0},
                },
                timeout=30,
            ).json()
            if r.get("status") == "ok":
                resultados.append(r)
            if len(resultados) >= n:
                break
        except Exception as exc:
            print(f"    [aviso] /semantic/process falhou: {exc}")
            break
    return resultados[:n]


def _to_img(tensor) -> np.ndarray:
    """Converte tensor (lista aninhada) em array numpy [H,W] ou [H,W,3]."""
    arr = np.array(tensor, dtype=np.float32)
    while arr.ndim > 3:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[0] in (1, 3):   # [C,H,W] → [H,W,C]
        arr = np.transpose(arr, (1, 2, 0))
    if arr.ndim == 3 and arr.shape[2] == 1:         # [H,W,1] → [H,W]
        arr = arr[:, :, 0]
    return np.clip(arr, 0.0, 1.0)


def _nome(idx) -> str:
    return LABEL_MAP.get(int(idx), str(idx))


def _veredito(clf: dict | None, chave: str) -> str:
    """Retorna '✓ avião (87%)' ou '✗ gato (61%)' para a chave dada."""
    if not clf or chave not in clf:
        return "sem juiz"
    d    = clf[chave]
    pred = _nome(d.get("pred", "?"))
    conf = d.get("confidence", 0.0) * 100
    icon = "✓" if d.get("correct_top1", False) else "✗"
    return f"{icon} {pred} ({conf:.0f}%)"


# ─── Loop principal ───────────────────────────────────────────────────────────
print("╔" + "═"*67 + "╗")
print("║  CÉLULA 6 — LABORATÓRIO VISUAL × LATENT_DIM (Figura do Artigo)   ║")
print("╚" + "═"*67 + "╝")
print(f"  Dataset: {DATASET}  |  Dims: {LATENT_DIMS}  |  Amostras/dim: {N_AMOSTRAS}")
print()

is_rgb = (DATASET == "cifar10")
cmap   = None if is_rgb else "gray"

queda_por_dim: dict[int, float | None] = {}

for dim in LATENT_DIMS:
    print(f"  ── latent_dim={dim} ─────────────────────────────────────────────")
    amostras = _buscar_amostras(dim, N_AMOSTRAS)

    if not amostras:
        print(f"    [aviso] Sem amostras para dim={dim}. Pesos treinados?")
        queda_por_dim[dim] = None
        continue

    # ── Calcular queda semântica deste dim ────────────────────────────────
    n_ok_orig  = sum(1 for a in amostras
                     if a.get("classifier", {}).get("original",      {}).get("correct_top1"))
    n_ok_recon = sum(1 for a in amostras
                     if a.get("classifier", {}).get("reconstructed", {}).get("correct_top1"))
    n = len(amostras)
    acc_o = n_ok_orig  / n
    acc_r = n_ok_recon / n
    queda = acc_o - acc_r
    queda_por_dim[dim] = queda
    print(f"    acc_original={acc_o:.1%}  acc_recon={acc_r:.1%}  ΔSemântica={queda:.1%}")

    # ── Montar mosaico ────────────────────────────────────────────────────
    # Colunas: Original | Recebida | Reconstruída | Info
    COLS = ["Original", "Recebida\n(ruído/máscara)", "Reconstruída\n(decoder)", "Métricas"]
    CHAVES_CLF = ["original", "received", "reconstructed", None]

    fig, axes = plt.subplots(n, 4, figsize=(4 * 2.8, n * 2.8),
                             gridspec_kw={"width_ratios": [1, 1, 1, 0.7]})
    fig.suptitle(
        f"{DATASET.upper()}  ·  latent_dim = {dim}"
        f"\nΔ Semântica = {queda:.1%}  "
        f"(orig={acc_o:.1%} → recon={acc_r:.1%})",
        fontsize=10, y=1.02,
    )

    # garante sempre array 2D de eixos mesmo com n=1
    if n == 1:
        axes = [axes]

    TITULOS = ["Original", "Recebida", "Reconstruída"]
    IMG_KEYS = ["original", "received", "reconstructed"]

    for row, am in enumerate(amostras):
        clf       = am.get("classifier", {})
        label_val = am.get("label", "?")
        psnr      = am.get("psnr", 0.0)
        ssim_v    = am.get("ssim", 0.0)

        for col in range(3):
            ax = axes[row][col]
            img = _to_img(am[IMG_KEYS[col]])
            ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(f"{TITULOS[col]}\n{_veredito(clf, CHAVES_CLF[col])}", fontsize=7, pad=2)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"GT: {_nome(label_val)}", fontsize=7, rotation=0,
                              labelpad=40, va="center")

        # 4ª coluna – painel de texto com métricas
        ax_m = axes[row][3]
        ax_m.axis("off")
        linhas = [
            f"GT:    {_nome(label_val)}",
            f"PSNR:  {psnr:.1f} dB",
            f"SSIM:  {ssim_v:.3f}",
            "",
            f"dim:   {dim}",
            f"SNR:   {AWGN_CFG['snr_db']} dB",
            f"mask:  {MASK_CFG['drop_rate']:.0%}",
        ]
        ax_m.text(0.05, 0.5, "\n".join(linhas), transform=ax_m.transAxes,
                  va="center", ha="left", fontsize=7.5, family="monospace")

    plt.tight_layout()
    fig_path = OUT_DIR / f"mosaico_dim{dim}.png"
    fig.savefig(fig_path, dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"    → Figura salva: {fig_path}")

# ─── Figura comparativa cross-dim ─────────────────────────────────────────────
print("\n  ── Gerando figura comparativa cross-dim ──")
dims_ok    = [d for d in LATENT_DIMS if queda_por_dim.get(d) is not None]
quedas_pct = [queda_por_dim[d] * 100 for d in dims_ok]  # type: ignore[operator]

if dims_ok:
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#161b22")

    cores = ["#ff6b6b" if q > 20 else "#ffd166" if q > 10 else "#00f6a2"
             for q in quedas_pct]
    barras = ax.bar([str(d) for d in dims_ok], quedas_pct,
                    color=cores, edgecolor="#30363d", linewidth=0.6)
    ax.axhline(5,  color="#00f6a2", linestyle="--", linewidth=1.2, label="limiar 5%")
    ax.axhline(20, color="#ff6b6b", linestyle="--", linewidth=1.2, label="limiar 20%")

    for b, q in zip(barras, quedas_pct):
        ax.text(b.get_x() + b.get_width() / 2, q + 0.4,
                f"{q:.1f}%", ha="center", va="bottom",
                fontsize=9, color="white", fontweight="bold")

    ax.set_title(f"Queda Semântica × Dimensão Latente — {DATASET.upper()}",
                 fontsize=12, color="white")
    ax.set_xlabel("latent_dim  (bytes transmitidos)", fontsize=10, color="white")
    ax.set_ylabel("Queda Semântica (%)", fontsize=10, color="white")
    ax.set_ylim(0, max(quedas_pct) * 1.3 + 5)
    ax.tick_params(colors="white")
    ax.legend(fontsize=9, facecolor="#161b22", labelcolor="white")
    ax.grid(True, axis="y", alpha=0.2, color="white")
    for sp in ax.spines.values():
        sp.set_edgecolor("#30363d")

    plt.tight_layout()
    cmp_path = OUT_DIR / "comparativo_queda_semantica.png"
    fig.savefig(cmp_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  → Figura comparativa: {cmp_path}")

# ─── Resumo terminal ─────────────────────────────────────────────────────────
print("\n" + "═"*68)
print("  RESUMO — CÉLULA 6")
print("═"*68)
print(f"  {'latent_dim':>12} | {'queda semântica':>16} | status")
print(f"  {'-'*12}-+-{'-'*16}-+--------")
for dim in LATENT_DIMS:
    q = queda_por_dim.get(dim)
    if q is None:
        status, q_str = "SEM DADOS", "N/A"
    elif q > 0.20:
        status, q_str = "⚠ ALTA",  f"{q*100:.1f}%"
    elif q > 0.05:
        status, q_str = "~ MÉDIA", f"{q*100:.1f}%"
    else:
        status, q_str = "✓ OK",    f"{q*100:.1f}%"
    print(f"  {dim:>12} | {q_str:>16} | {status}")

print(f"\n✓ Figuras em: {OUT_DIR}")
print("  Execute fix_cell7.py para gerar a Curva de Trade-off completa.")

# Persiste para cell7
Path("shared_data/resultados/notebook_monitor").mkdir(parents=True, exist_ok=True)
Path("shared_data/resultados/notebook_monitor/cell6_resultado.json").write_text(
    json.dumps({
        "dataset":        DATASET,
        "model":          MODEL,
        "latent_dims":    LATENT_DIMS,
        "queda_semantica": {str(k): v for k, v in queda_por_dim.items()},
    }, indent=2),
    encoding="utf-8",
)


---
## 🏭 Módulo 4: A Fábrica de Trabalhos (Triple Trade-off)

O Evento Principal. Estressando as taxas de compactação a níveis drásticos para exportar pontuações de Acurácia x Compressão para o Manuscripto Final acadêmico.


In [ ]:
"""
fix_cell7.py — Célula 7 (Arquitetura Final Unificada)
======================================================
Para cada latent_dim em [16, 32, 64, 128, 256]:
  - Carrega encoder_{dim}.pth  (salvo na Célula 4)
  - Carrega classifier_{dim}.pth (salvo na Célula 5)
  - Roda benchmark: mede compressão × queda semântica
  - ZERO treinamento — apenas inferência

Saídas:
  ▸ benchmark_tradeoff.csv       — tabela para LaTeX
  ▸ curva_tradeoff.png           — figura principal do artigo
  ▸ curva_tradeoff_paper.png     — versão limpa publication-ready

Execute com:
  %run fix_cell7.py
"""

import csv
import json
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import requests

matplotlib.rcParams.update({
    "font.family":    "DejaVu Sans",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

# ─── Configuração ─────────────────────────────────────────────────────────────
ML_SERVICE  = "http://localhost:8000"
DATASET     = "cifar10"        # mnist | fashion | cifar10
MODEL       = "cnn_vae"
LATENT_DIMS = [256]
N_AMOSTRAS  = 50               # amostras por dim (mais = mais preciso)

AWGN_CFG = {"enabled": True,  "snr_db": 15}
MASK_CFG = {"enabled": True,  "drop_rate": 0.1, "fill_value": 0.0}

OUT_DIR = Path("shared_data/resultados/benchmark")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Bytes brutos por dataset (original sem compressão)
_BYTES_BRUTOS = {"mnist": 3136, "fashion": 3136, "cifar10": 12288}
BYTES_ORIGINAL = _BYTES_BRUTOS.get(DATASET, 12288)


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _benchmark_dim(latent_dim: int, n: int) -> dict | None:
    """
    Chama /experiment/benchmark com include_samples=True para um único latent_dim.
    Retorna dict com métricas agregadas ou None em caso de falha.
    """
    try:
        r = requests.post(
            f"{ML_SERVICE}/experiment/benchmark",
            json={
                "datasets":       [DATASET],
                "models":         [MODEL],
                "latent_dim":     latent_dim,
                "bits":           8,
                "num_samples":    n,
                "seed":           42,
                "awgn":           AWGN_CFG,
                "masking":        MASK_CFG,
                "classifier":     {"enabled": True, "top_k": 1, "min_confidence": 0.0},
                "include_samples": False,
            },
            timeout=120,
        )
        data = r.json()
    except Exception as exc:
        print(f"    [ERRO HTTP] dim={latent_dim}: {exc}")
        return None

    # Extrai o bloco do dataset/model correto
    for res in data.get("results", []):
        if res.get("dataset") == DATASET and res.get("model") == MODEL:
            if res.get("status") == "error":
                print(f"    [ERRO benchmark] dim={latent_dim}: {res.get('error')}")
                return None
            return res

    return None


# ─── Loop de benchmark ────────────────────────────────────────────────────────
print("╔" + "═"*67 + "╗")
print("║  CÉLULA 7 — BENCHMARK TRADE-OFF (Curva do Paper)                  ║")
print("╚" + "═"*67 + "╝")
print(f"  Dataset: {DATASET}  |  Modelo: {MODEL}")
print(f"  Dims:    {LATENT_DIMS}")
print(f"  Amostras/dim: {N_AMOSTRAS}  |  AWGN={AWGN_CFG}  |  mask={MASK_CFG}")
print()

linhas_tabela: list[dict] = []

for dim in LATENT_DIMS:
    print(f"  ── latent_dim={dim} ─────────────────────────────────────────────")
    t0  = time.time()
    res = _benchmark_dim(dim, N_AMOSTRAS)
    dt  = time.time() - t0

    if res is None:
        print(f"    ✗ falhou ({dt:.1f}s)")
        linhas_tabela.append({
            "latent_dim": dim, "status": "erro",
            "bytes_latente": dim * 1,       # 1 byte/dim @ 8bits
            "taxa_compressao": None,
            "reducao_bw_pct": None,
            "mse_mean": None, "psnr_mean": None, "ssim_mean": None,
            "acc_original": None, "acc_recon": None,
            "queda_semantica_pct": None,
        })
        continue

    # ── Extrai métricas ───────────────────────────────────────────────────
    latent_bytes   = res.get("latent_bytes",           dim)          # bytes @ 8bits
    comp_ratio     = res.get("compression_ratio_mean", BYTES_ORIGINAL / latent_bytes)
    reducao_bw     = res.get("bandwidth_reduction_pct",
                             (1 - latent_bytes / BYTES_ORIGINAL) * 100)
    mse            = res.get("mse_mean",  None)
    psnr           = res.get("psnr_mean", None)
    ssim_v         = res.get("ssim_mean", None)
    clf            = res.get("classification", {})
    acc_orig       = clf.get("accuracy_original",      None)
    acc_recon      = clf.get("accuracy_reconstructed", None)
    queda          = (acc_orig - acc_recon) * 100 if (acc_orig is not None and acc_recon is not None) else None

    linha = {
        "latent_dim":          dim,
        "status":              "ok",
        "bytes_latente":       latent_bytes,
        "taxa_compressao":     round(comp_ratio, 2)  if comp_ratio is not None else None,
        "reducao_bw_pct":      round(reducao_bw, 1)  if reducao_bw is not None else None,
        "mse_mean":            round(mse,    5)       if mse    is not None else None,
        "psnr_mean":           round(psnr,   2)       if psnr   is not None else None,
        "ssim_mean":           round(ssim_v, 4)       if ssim_v is not None else None,
        "acc_original":        round(acc_orig,  4)    if acc_orig  is not None else None,
        "acc_recon":           round(acc_recon, 4)    if acc_recon is not None else None,
        "queda_semantica_pct": round(queda, 2)        if queda is not None else None,
    }
    linhas_tabela.append(linha)

    # ── Impressão resumida ────────────────────────────────────────────────
    print(f"    bytes_latente={latent_bytes}  compressão={comp_ratio:.1f}×  -BW={reducao_bw:.1f}%")
    print(f"    PSNR={psnr:.2f}dB  SSIM={ssim_v:.3f}  MSE={mse:.5f}")
    if queda is not None:
        print(f"    acc_orig={acc_orig:.1%}  acc_recon={acc_recon:.1%}  Δsem={queda:.1f}%  ({dt:.1f}s)")
    else:
        print(f"    [sem classificador]  ({dt:.1f}s)")


# ─── Exportar CSV ─────────────────────────────────────────────────────────────
csv_path = OUT_DIR / "benchmark_tradeoff.csv"
if linhas_tabela:
    campos = list(linhas_tabela[0].keys())
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(linhas_tabela)
    print(f"\n  → CSV salvo: {csv_path}")


# ─── Exportar tabela LaTeX ────────────────────────────────────────────────────
tex_path = OUT_DIR / "benchmark_tradeoff.tex"
with open(tex_path, "w", encoding="utf-8") as f:
    f.write("% Tabela gerada automaticamente por fix_cell7.py\n")
    f.write("\\begin{table}[h]\n")
    f.write("\\centering\n")
    f.write("\\caption{Trade-off Compressão × Semântica — " + DATASET.upper() + "}\n")
    f.write("\\label{tab:tradeoff}\n")
    f.write("\\begin{tabular}{ccccccc}\n")
    f.write("\\hline\n")
    f.write("$d_{latente}$ & Bytes & Compressão & $\\Delta$BW (\\%) & "
            "PSNR (dB) & SSIM & $\\Delta$Sem. (\\%) \\\\\n")
    f.write("\\hline\n")
    for l in linhas_tabela:
        if l["status"] != "ok":
            continue
        f.write(
            f"{l['latent_dim']} & {l['bytes_latente']} & "
            f"{l['taxa_compressao']:.1f}$\\times$ & "
            f"{l['reducao_bw_pct']:.1f} & "
            f"{l['psnr_mean']:.2f} & "
            f"{l['ssim_mean']:.3f} & "
            f"{l['queda_semantica_pct'] if l['queda_semantica_pct'] is not None else 'N/A'} \\\\\n"
        )
    f.write("\\hline\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")
print(f"  → LaTeX salvo: {tex_path}")


# ─── Gráficos ─────────────────────────────────────────────────────────────────
dims_ok   = [l for l in linhas_tabela if l["status"] == "ok"]
x_dims    = [l["latent_dim"]          for l in dims_ok]
y_comp    = [l["taxa_compressao"]     for l in dims_ok]
y_bw      = [l["reducao_bw_pct"]     for l in dims_ok]
y_queda   = [l["queda_semantica_pct"] for l in dims_ok]
y_psnr    = [l["psnr_mean"]          for l in dims_ok]
y_ssim    = [l["ssim_mean"]          for l in dims_ok]

has_sem = any(q is not None for q in y_queda)

# ── Figura 1: Trade-off completo (4 painéis) ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"Benchmark Trade-off — {DATASET.upper()} / {MODEL.upper()}", fontsize=13)
fig.patch.set_facecolor("#0d1117")

paleta = "#489dff"

for ax in axes.flat:
    ax.set_facecolor("#161b22")
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for sp in ax.spines.values():
        sp.set_edgecolor("#30363d")
    ax.grid(True, alpha=0.2, color="white")

# ── [0,0] Taxa de Compressão × latent_dim
ax = axes[0, 0]
ax.plot(x_dims, y_comp, color=paleta, marker="o", linewidth=2)
ax.set_title("Taxa de Compressão")
ax.set_xlabel("latent_dim")
ax.set_ylabel("Compressão (×)")
ax.fill_between(x_dims, y_comp, alpha=0.15, color=paleta)

# ── [0,1] Redução de Largura de Banda
ax = axes[0, 1]
ax.plot(x_dims, y_bw, color="#00f6a2", marker="s", linewidth=2)
ax.set_title("Redução de Largura de Banda")
ax.set_xlabel("latent_dim")
ax.set_ylabel("Redução BW (%)")
ax.fill_between(x_dims, y_bw, alpha=0.15, color="#00f6a2")

# ── [1,0] PSNR / SSIM
ax = axes[1, 0]
ax2 = ax.twinx()
ax.plot(x_dims,  y_psnr, color="#ffd166", marker="^", linewidth=2, label="PSNR (dB)")
ax2.plot(x_dims, y_ssim, color="#ff6b6b", marker="v", linewidth=2, linestyle="--", label="SSIM")
ax.set_title("Qualidade da Reconstrução")
ax.set_xlabel("latent_dim")
ax.set_ylabel("PSNR (dB)", color="#ffd166")
ax2.set_ylabel("SSIM",     color="#ff6b6b")
ax.tick_params(axis="y", colors="#ffd166")
ax2.tick_params(axis="y", colors="#ff6b6b")
ax2.set_facecolor("#161b22")
for sp in ax2.spines.values():
    sp.set_edgecolor("#30363d")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, facecolor="#161b22",
          labelcolor="white", fontsize=8)

# ── [1,1] Queda Semântica
ax = axes[1, 1]
if has_sem:
    q_vals = [q if q is not None else 0 for q in y_queda]
    cores  = ["#ff6b6b" if q > 20 else "#ffd166" if q > 10 else "#00f6a2"
              for q in q_vals]
    barras = ax.bar([str(d) for d in x_dims], q_vals, color=cores,
                    edgecolor="#30363d", linewidth=0.6)
    ax.axhline(5,  color="#00f6a2", linestyle="--", linewidth=1.2, label="5%")
    ax.axhline(20, color="#ff6b6b", linestyle="--", linewidth=1.2, label="20%")
    for b, q in zip(barras, q_vals):
        ax.text(b.get_x() + b.get_width() / 2, q + 0.3,
                f"{q:.1f}%", ha="center", va="bottom",
                fontsize=8, color="white", fontweight="bold")
    ax.set_ylim(0, max(q_vals) * 1.35 + 5)
    ax.legend(facecolor="#161b22", labelcolor="white", fontsize=8,
              title="limiares", title_fontsize=7)
else:
    ax.text(0.5, 0.5, "Classificador não carregado\n(execute fix_cell5.py)",
            ha="center", va="center", color="white", fontsize=10,
            transform=ax.transAxes)
ax.set_title("Queda Semântica (Δ%)")
ax.set_xlabel("latent_dim")
ax.set_ylabel("Δ Semântica (%)")

plt.tight_layout()
fig1_path = OUT_DIR / "curva_tradeoff.png"
fig.savefig(fig1_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"\n  → Gráfico trade-off: {fig1_path}")


# ── Figura 2: Publication-ready (2 painéis lado a lado) ──────────────────────
fig, (ax_bw, ax_sem) = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle(
    f"Semantic Compression Trade-off — {DATASET.upper()}",
    fontsize=12, fontweight="bold",
)

# Painel esquerdo: BW reduction
ax_bw.plot(x_dims, y_bw, color="#489dff", marker="o", linewidth=2.5,
           markersize=7, label="BW Reduction (%)")
ax_bw.fill_between(x_dims, y_bw, alpha=0.12, color="#489dff")
ax_bw.set_xlabel("Latent Dimension (bytes)")
ax_bw.set_ylabel("Bandwidth Reduction (%)")
ax_bw.set_title("Compression Efficiency")
ax_bw.grid(True, alpha=0.3)
ax_bw.legend()

# Painel direito: Queda semântica
if has_sem:
    q_vals = [q if q is not None else 0 for q in y_queda]
    ax_sem.plot(x_dims, q_vals, color="#ff6b6b", marker="s", linewidth=2.5,
                markersize=7, label="Semantic Drop (%)")
    ax_sem.fill_between(x_dims, q_vals, alpha=0.12, color="#ff6b6b")
    ax_sem.axhline(5,  color="#00b894", linestyle="--", linewidth=1.2, label="5% threshold")
    ax_sem.axhline(20, color="#d63031", linestyle="--", linewidth=1.2, label="20% threshold")
    ax_sem.set_ylim(bottom=0)

    # Zona segura
    ax_sem.axhspan(0, 5, alpha=0.06, color="#00b894", label="safe zone")
    ax_sem.legend(fontsize=8)
else:
    ax_sem.text(0.5, 0.5, "No classifier data\n(run fix_cell5.py)",
                ha="center", va="center", fontsize=11, transform=ax_sem.transAxes)

ax_sem.set_xlabel("Latent Dimension (bytes)")
ax_sem.set_ylabel("Semantic Drop (%)")
ax_sem.set_title("Semantic Preservation")
ax_sem.grid(True, alpha=0.3)

plt.tight_layout()
fig2_path = OUT_DIR / "curva_tradeoff_paper.png"
fig.savefig(fig2_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"  → Versão paper: {fig2_path}")


# ─── Resumo terminal ──────────────────────────────────────────────────────────
print("\n" + "═"*78)
print("  RESUMO FINAL — CÉLULA 7")
print("═"*78)
hdr = f"{'dim':>6} | {'bytes':>6} | {'comp':>6} | {'BW%':>6} | "
hdr += f"{'PSNR':>7} | {'SSIM':>6} | {'acc_orig':>9} | {'acc_recon':>9} | {'Δsem%':>7}"
print(f"  {hdr}")
print(f"  {'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*7}-+-"
      f"{'-'*6}-+-{'-'*9}-+-{'-'*9}-+-{'-'*7}")
for l in linhas_tabela:
    if l["status"] != "ok":
        print(f"  {l['latent_dim']:>6} | ERRO")
        continue
    ao = f"{l['acc_original']*100:.1f}%"  if l["acc_original"]  is not None else "N/A"
    ar = f"{l['acc_recon']*100:.1f}%"     if l["acc_recon"]     is not None else "N/A"
    qs = f"{l['queda_semantica_pct']:.1f}%" if l["queda_semantica_pct"] is not None else "N/A"
    print(
        f"  {l['latent_dim']:>6} | {l['bytes_latente']:>6} | "
        f"{l['taxa_compressao']:>5.1f}× | "
        f"{l['reducao_bw_pct']:>5.1f}% | "
        f"{l['psnr_mean']:>7.2f} | "
        f"{l['ssim_mean']:>6.3f} | "
        f"{ao:>9} | {ar:>9} | {qs:>7}"
    )

print(f"\n  Arquivos gerados em: {OUT_DIR}")
print("  ▸ benchmark_tradeoff.csv")
print("  ▸ benchmark_tradeoff.tex")
print("  ▸ curva_tradeoff.png")
print("  ▸ curva_tradeoff_paper.png")
print("\n✓ Benchmark concluído. Pipeline completo!\n")

# Persiste JSON para referência futura
Path("shared_data/resultados/notebook_monitor").mkdir(parents=True, exist_ok=True)
Path("shared_data/resultados/notebook_monitor/cell7_resultado.json").write_text(
    json.dumps({
        "dataset": DATASET, "model": MODEL,
        "latent_dims": LATENT_DIMS,
        "n_amostras":  N_AMOSTRAS,
        "resultados":  linhas_tabela,
    }, indent=2),
    encoding="utf-8",
)
